In [1]:
!pip install fastembed chromadb groq pypdf2 ipywidgets -q

In [2]:
import os
import chromadb
import PyPDF2
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from fastembed import TextEmbedding
from groq import Groq
import io

✅ All imports done


In [9]:
GROQ_API_KEY = "XXXXXXXXXXXXXXXXXX"
print('Loading embedding model (130MB, first time only)...')
embedding_model = TextEmbedding('BAAI/bge-small-en-v1.5')
groq_client = Groq(api_key=GROQ_API_KEY)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name='resume_db')
uploaded_resumes = [] 

Loading embedding model (130MB, first time only)...
✅ Models ready!


In [10]:
def extract_text_from_pdf(file_bytes):
    reader = PyPDF2.PdfReader(io.BytesIO(file_bytes))
    text = ' '.join(
        page.extract_text() for page in reader.pages
        if page.extract_text()
    )
    return text.strip()
def store_resume(filename, content):
    doc_id = f"resume_{filename.replace(' ', '_')}"
    existing = collection.get(ids=[doc_id])
    if existing['ids']:
        collection.delete(ids=[doc_id])
    embedding = list(embedding_model.embed([content]))[0].tolist()
    collection.add(
        ids=[doc_id],
        embeddings=[embedding],
        documents=[content],
        metadatas=[{'filename': filename}]
    )
def retrieve(query, top_k=3):
    count = collection.count()
    if count == 0:
        return None
    query_embedding = list(embedding_model.embed([query]))[0].tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, count)
    )
    chunks = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        chunks.append(f"[{meta['filename']}]\n{doc}")
    return '\n\n'.join(chunks)

def ask_groq(question):
    context = retrieve(question)
    if not context:
        return '⚠️ No resumes uploaded yet!'

    prompt = f"""You are a resume analysis assistant.
Use ONLY the resume content below to answer the question.
Be specific — mention candidate names and details.

Resume Context:
{context}

Question: {question}

Answer:"""

    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

✅ Functions ready!


In [11]:
def clean_text(text):
    return text.encode('ascii', 'ignore').decode('ascii').strip()
def extract_text_from_pdf(file_bytes):
    reader = PyPDF2.PdfReader(io.BytesIO(file_bytes))
    text = ' '.join(
        page.extract_text() for page in reader.pages
        if page.extract_text()
    )
    return clean_text(text) 

In [12]:
upload_widget = widgets.FileUpload(
    accept='.pdf',
    multiple=True,
    description='📂 Upload PDFs',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

upload_btn = widgets.Button(
    description='Store Resumes',
    button_style='primary',
    icon='upload',
    layout=widgets.Layout(width='160px', height='36px')
)

upload_status = widgets.Output()
resume_list_out = widgets.Output()
question_box = widgets.Text(
    placeholder='e.g. Who has the highest CGPA?',
    description='❓ Question:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
ask_btn = widgets.Button(
    description='Ask',
    button_style='success',
    icon='search',
    layout=widgets.Layout(width='100px', height='36px')
)
answer_out = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ccc',
        padding='12px',
        min_height='80px',
        width='580px'
    )
)
def on_upload_click(b):
    with upload_status:
        clear_output()
        if not upload_widget.value:
            print('⚠️ No files selected!')
            return
        for file_info in upload_widget.value:
            filename = file_info['name']
            content_bytes = file_info['content']
            try:
                text = extract_text_from_pdf(bytes(content_bytes))
                if not text:
                    print(f'⚠️ {filename} — could not extract text (scanned PDF?)')
                    continue
                store_resume(filename, text)
                if filename not in uploaded_resumes:
                    uploaded_resumes.append(filename)
                print(f'✅ {filename} stored!')
            except Exception as e:
                print(f'❌ {filename} failed: {e}')
    with resume_list_out:
        clear_output()
        if uploaded_resumes:
            print(f'📋 Stored resumes ({len(uploaded_resumes)}):')
            for r in uploaded_resumes:
                print(f'   • {r}')
def on_ask_click(b):
    with answer_out:
        clear_output()
        q = question_box.value.strip()
        if not q:
            print('⚠️ Type a question first!')
            return
        print('⏳ Thinking...')
        clear_output(wait=True)
        answer = ask_groq(q)
        print(answer)

upload_btn.on_click(on_upload_click)
ask_btn.on_click(on_ask_click)
display(HTML('<h3>📄 Resume RAG System</h3>'))
display(HTML('<b>Step 1 — Upload Resumes</b>'))
display(widgets.HBox([upload_widget, upload_btn]))
display(upload_status)
display(resume_list_out)
display(HTML('<br><b>Step 2 — Ask a Question</b>'))
display(widgets.HBox([question_box, ask_btn]))
display(HTML('<br><b>Answer:</b>'))
display(answer_out)

Output()

Output()

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…